#  Data Wrangling dan Preprocessing TripWell
- **ID Tim:** CC26-PSU116

## Menentukan Pertanyaan Bisnis

### Pertanyaan Bisnis
- **Pertanyaan 1:**  Lokasi wisata alam mana di Bandung Barat yang memiliki proporsi ulasan aksesibilitas negatif tertinggi berdasarkan persentase label negatif terhadap total ulasan per lokasi, dan bagaimana rata-rata rating pada lokasi terkait selama periode 2019-2023?
- **Pertanyaan 2:** Apa saja kata dan bigram paling diskriminatif untuk  membedakan ulasan berlabel aksesibilitas positif dan negatif berdasarkan nilai TF-IDF dalam dataset pada tahun 2019–2023?
- **Pertanyaan 3:** Bagaimana proporsi bulanan label aksesibilitas negatif, netral, dan positif berubah sepanjang 2019 –  2023, dan apakah terdapat tren atau pola musiman?



In [3]:
# Import Semua Packages/Library yang Digunakan

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random
import json
from pathlib import Path



#Untuk Text
import nltk
import re
import string
from bs4 import BeautifulSoup
from nltk.tokenize import word_tokenize
from tqdm import trange
from nltk import tokenize
from nltk.probability import FreqDist
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer

## Data Wrangling

### Gathering Data

Data yang digunakan diambil dari **https://www.kaggle.com/datasets/dzzlr07/absa-natural-tourist-attractions-review**

In [4]:
df = pd.read_csv("tourist_reviews.csv", delimiter=",")

df.head(5)

,id,datetime,location,text,rating,accessibility,facility,activity
0,47d0cdd937754bd6b860f89b2bab1dbb,2022-05-15 11:58:43,Curug Malela,Akses jalannya waktu itu masih sulit di jangka...,4,negative,neutral,neutral
1,4804acd6c05e4f89b098e2ca35019419,2022-08-15 11:58:43,Curug Malela,"Perjalanan yg bnr"" bikin Syahduu ,, dr Tempat ...",5,neutral,neutral,positive
2,3eae265bf32a45eca31765a4145bc030,2022-03-15 11:58:43,Curug Malela,"Minggu 13 februari 2022 ,\n\ngas santai pakai ...",5,positive,negative,positive
3,61037dbdb7b14045be49d4494e95cf05,2022-05-15 11:58:44,Curug Malela,7 mei 2022\nTouring bari mudik\nMntap perjalan...,5,positive,neutral,positive
4,a2c9e817e2b949c6880f971f43a11d2f,2022-08-15 11:58:44,Curug Malela,Perjalanan touring motor dari bekasi melewati ...,5,neutral,neutral,positive


### Assessing Data

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11620 entries, 0 to 11619
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   id             11620 non-null  object
 1   datetime       11620 non-null  object
 2   location       11620 non-null  object
 3   text           11620 non-null  object
 4   rating         11620 non-null  int64 
 5   accessibility  11620 non-null  object
 6   facility       11620 non-null  object
 7   activity       11620 non-null  object
dtypes: int64(1), object(7)
memory usage: 726.4+ KB


In [6]:
df.describe(include="all")

,id,datetime,location,text,rating,accessibility,facility,activity
count,11620,11620,11620,11620,11620.000000,11620,11620,11620
unique,11620,5828,25,11620,NaN,3,3,3
top,821d8c4e3513424bbdc395f4c32b72f5,2021-02-15 11:58:47,Sarae Hills,Tempat dan suasananya sejuk bangeeettt....,NaN,neutral,neutral,positive
freq,1,11,952,1,NaN,10197,8105,7153
mean,NaN,NaN,NaN,NaN,4.502582,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,0.833755,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,1.000000,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,4.000000,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,5.000000,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,5.000000,NaN,NaN,NaN


In [7]:
print("Jumlah duplikasi struktural : ", df.duplicated().sum())

Jumlah duplikasi struktural :  0


In [8]:
# Distribusi Label Sentimen Aksesibilitas, Fasilitas, dan Aktivitas

# Kolom label yang ingin dicek
label_cols = ["accessibility", "facility", "activity"]

# Cek distribusi kelas
for col in label_cols:
    print(f"\n=== Class Distribution: {col.upper()} ===")

    # Hitung jumlah dan proporsi
    class_count = df[col].value_counts()
    class_ratio = df[col].value_counts(normalize=True) * 100

    balance_df = pd.DataFrame({
        "count": class_count,
        "percentage": class_ratio.round(2)
    })

    print(balance_df)


=== Class Distribution: ACCESSIBILITY ===
               count  percentage
accessibility                   
neutral        10197       87.75
negative         862        7.42
positive         561        4.83

=== Class Distribution: FACILITY ===
          count  percentage
facility                   
neutral    8105       69.75
positive   2318       19.95
negative   1197       10.30

=== Class Distribution: ACTIVITY ===
          count  percentage
activity                   
positive   7153       61.56
neutral    4084       35.15
negative    383        3.30


In [9]:
#Assessing fitur 'text'

#@ Menghitung jumlah kata unik dan panjang teks maksimum
all_words = ' '.join(df['text'].dropna()).split()
max_words = len(set(all_words))

print(f"Jumlah kata unik (max_words): {max_words}")

df['word_count'] = df['text'].apply(lambda x: len(str(x).split()) if x is not None else 0)

max_seq_len = df['word_count'].max()

print(f"Panjang kalimat maksimum (max_length): {max_seq_len} kata")

# Cek kemunculan angka
print("Cek Kemunculan Angka")
df['text'].str.contains(r'\d').sum()

# Cek ada HTML tag tidak
print("Cek HTML Tag")
df['text'].str.contains(r'<[^>]+>').sum()

# Cek ada URL tidak
print("Cek kemunculan URL")
df['text'].str.contains(r'http|www').sum()

# Cek teks yang sangat pendek (kemungkinan tidak bermakna)
df['text'].str.split().str.len().value_counts().head(10)

# Karakter non-latin yang mencurigakan
df['text'].str.contains(r'[^\x00-\x7F]').sum()


Jumlah kata unik (max_words): 35195
Panjang kalimat maksimum (max_length): 567 kata
Cek Kemunculan Angka
Cek HTML Tag
Cek kemunculan URL


np.int64(1382)

**Insights and Steps to Take:**
- Tidak ada missing value, duplicate row secara struktural, maupun nilai numerik yang mencurigakan
- Fitur `datetime` seharusnya bertipe datetime bukan object sehingga perlu convert tipe data
- Untuk  komputasi perlu dilakukan encoding fitur kategorik `accessibility`, `facility`, dan `activity` dengan label encoding karena ketiga fitur ini merupakan label  multiclass dengan urutan ordinal (negative < neutral < positive).
- Perlu dilakukan preprocessing terhadap fitur `text` agar analisis topik dan klasifikasi aspek aksesibilitas dapat dilakukan secara lebih akurat.
- Tidak akan dilakukan stopwords removal karena akan menghilangkan kata-kata penting untuk klasifikasi sentimen.

### Cleaning Data

In [10]:
# mengubah tipe object ke datetime
df["datetime"] = pd.to_datetime(df["datetime"])

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11620 entries, 0 to 11619
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   id             11620 non-null  object        
 1   datetime       11620 non-null  datetime64[ns]
 2   location       11620 non-null  object        
 3   text           11620 non-null  object        
 4   rating         11620 non-null  int64         
 5   accessibility  11620 non-null  object        
 6   facility       11620 non-null  object        
 7   activity       11620 non-null  object        
 8   word_count     11620 non-null  int64         
dtypes: datetime64[ns](1), int64(2), object(6)
memory usage: 817.2+ KB


In [11]:
# Label Encoding untuk fitur accessibility, activity, dan facility

# negative < neutral < positive

label_map = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

# Encode kolom target
target_cols = ["accessibility", "facility", "activity"]

for col in target_cols:
    df[col] = df[col].map(label_map)

# Cek hasil encoding
df[["accessibility", "facility", "activity"]].head()

,accessibility,facility,activity
0,0,1,1
1,1,1,2
2,2,0,2
3,2,1,2
4,1,1,2


In [12]:
df.head()

,id,datetime,location,text,rating,accessibility,facility,activity,word_count
0,47d0cdd937754bd6b860f89b2bab1dbb,2022-05-15 11:58:43,Curug Malela,Akses jalannya waktu itu masih sulit di jangka...,4,0,1,1,13
1,4804acd6c05e4f89b098e2ca35019419,2022-08-15 11:58:43,Curug Malela,"Perjalanan yg bnr"" bikin Syahduu ,, dr Tempat ...",5,1,1,2,23
2,3eae265bf32a45eca31765a4145bc030,2022-03-15 11:58:43,Curug Malela,"Minggu 13 februari 2022 ,\n\ngas santai pakai ...",5,2,0,2,349
3,61037dbdb7b14045be49d4494e95cf05,2022-05-15 11:58:44,Curug Malela,7 mei 2022\nTouring bari mudik\nMntap perjalan...,5,2,1,2,26
4,a2c9e817e2b949c6880f971f43a11d2f,2022-08-15 11:58:44,Curug Malela,Perjalanan touring motor dari bekasi melewati ...,5,1,1,2,22


In [13]:
# Preprocessing Text

# slang dictionary
slang_dict = {
    "gk": "tidak",
    "ga": "tidak",
    "gak": "tidak",
    "nggak": "tidak",
    "tdk": "tidak",
    "bgt": "banget",
    "bngt": "banget",
    "bnr": "benar",
    "sy": "saya",
    "gw": "saya",
    "gue": "saya",
    "dr": "dari",
    "mntap": "mantap",
    "mantul": "mantap",
    "yg": "yang",
    "utk": "untuk",
    "jd": "jadi",
    "jg": "juga",
    "jln": "jalan",
    "krn": "karena",
    "dgn": "dengan",
    "blm": "belum",
    "sdh": "sudah",
    "msh": "masih",
    "aja": "saja",
    "emang": "memang",
    "trs": "terus",
    "dpt": "dapat",
    "sm": "sama",
    "kmrn": "kemarin",
    "tp": "tapi",
    "klo": "kalau",
    "kl": "kalau",
    "udh": "sudah",
    "udah": "sudah",
    "bs": "bisa",
    "byk": "banyak",
    "lg": "lagi",
    "lbh": "lebih",
}

def clean_text(text):
    text = str(text)

    # 1. lowercase
    text = text.lower()

    # 2. hapus html tags
    text = BeautifulSoup(text, "html.parser").get_text()

    # 3. hapus url
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)

    # 4. simpan angka+unit, hapus angka biasa
    units = r"(?:km|meter|menit|jam|rb|ribu|kg)"
    text = re.sub(r"(\d[\d\-\.]*\s*" + units + r")", r" \1 ", text)  # isolasi angka+unit
    text = re.sub(r"\b\d+\b", " ", text)                              # hapus angka tanpa unit

    # 5. hapus emoji dan karakter non-latin
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)

    # 6. normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    # 7. tokenisasi + slang normalization
    tokens = text.split()
    tokens = [slang_dict.get(word, word) for word in tokens]

    # 8. join kembali
    text = " ".join(tokens)
    return text if len(text.split()) >= 2 else None #filter teks yang terlalu pendek setelah preprocessing


df["text_clean"] = df["text"].apply(clean_text)

# cek hasil
null_after = df["text_clean"].isnull().sum()
print(f"Teks terlalu pendek / null setelah cleaning: {null_after}")

#preview
df.head()

Teks terlalu pendek / null setelah cleaning: 104


,id,datetime,location,text,rating,accessibility,facility,activity,word_count,text_clean
0,47d0cdd937754bd6b860f89b2bab1dbb,2022-05-15 11:58:43,Curug Malela,Akses jalannya waktu itu masih sulit di jangka...,4,0,1,1,13,akses jalannya waktu itu masih sulit di jangka...
1,4804acd6c05e4f89b098e2ca35019419,2022-08-15 11:58:43,Curug Malela,"Perjalanan yg bnr"" bikin Syahduu ,, dr Tempat ...",5,1,1,2,23,perjalanan yang benar bikin syahduu dari tempa...
2,3eae265bf32a45eca31765a4145bc030,2022-03-15 11:58:43,Curug Malela,"Minggu 13 februari 2022 ,\n\ngas santai pakai ...",5,2,0,2,349,minggu februari gas santai pakai motor dari da...
3,61037dbdb7b14045be49d4494e95cf05,2022-05-15 11:58:44,Curug Malela,7 mei 2022\nTouring bari mudik\nMntap perjalan...,5,2,1,2,26,mei touring bari mudik mantap perjalanan sungg...
4,a2c9e817e2b949c6880f971f43a11d2f,2022-08-15 11:58:44,Curug Malela,Perjalanan touring motor dari bekasi melewati ...,5,1,1,2,22,perjalanan touring motor dari bekasi melewati ...


In [14]:
# Simpan file
df.to_csv("preprocessed_dataset.csv", index=False)

**Insights and Steps to Take:**
- Fitur `datetime` sudah dikonversi menjadi bertipe datetime bukan objects
- telah dilakukan encoding fitur kategorik `accessibility`, `facility`, dan `activity` dengan label encoding.
- Telah dilakukan preprocessing (cleaning dan slang normalization) terhadap fitur `text`.
- Karena label **neutral** pada fitur `accesibility` tidak bermakna maka saat modelling label ini tidak digunakan.
- Lebih lanjut, karena data berlabel **positive** berjumlah 561 dan data berlabel **negative** berjumlah 862 maka saat modelling wajib dilakukan augmentasi data dengan synonym swap **hanya pada data train**.
- Tidak dilakukan filtering label positif dan negatif karena `preprocessed_dataset.csv` akan digunakan untuk EDA dan dashboard.

## Pembuatan Data Dictionary

In [15]:
CSV_FILE = "preprocessed_dataset.csv"
OUTPUT_JSON = "data_dictionary.json"

# Load CSV
df = pd.read_csv(CSV_FILE)

# PREPROCESSING METADATA
# Simpan informasi preprocessing
preprocessing_metadata = {
    "accessibility": {
        "method": "label_encoding",
        "original_type": "string",
        "processed_type": "integer",
        "mapping": {
            "negative": 0,
            "neutral": 1,
            "positive": 2
        },
        "description": "Accessibility sentiment label"
    },

    "facility": {
        "method": "label_encoding",
        "original_type": "string",
        "processed_type": "integer",
        "mapping": {
            "negative": 0,
            "neutral": 1,
            "positive": 2
        },
        "description": "Facility sentiment label"
    },

    "activity": {
        "method": "label_encoding",
        "original_type": "string",
        "processed_type": "integer",
        "mapping": {
            "negative": 0,
            "neutral": 1,
            "positive": 2
        },
        "description": "Activity sentiment label"
    }
}

# Buat Data Dictionary
data_dictionary = {
    "file_name": Path(CSV_FILE).name,
    "num_rows": int(len(df)),
    "num_columns": int(len(df.columns)),
    "columns": []
}

for col in df.columns:

    column_info = {
        "column_name": col,
        "data_type": str(df[col].dtype),
        "nullable": bool(df[col].isnull().any()),
        "null_count": int(df[col].isnull().sum()),
        "unique_count": int(df[col].nunique()),
        "sample_values": df[col].dropna().astype(str).head(5).tolist()
    }

    # Statistik data Numerik
    if pd.api.types.is_numeric_dtype(df[col]):

        column_info["min"] = (
            float(df[col].min())
            if not df[col].isnull().all()
            else None
        )

        column_info["max"] = (
            float(df[col].max())
            if not df[col].isnull().all()
            else None
        )

        column_info["mean"] = (
            float(df[col].mean())
            if not df[col].isnull().all()
            else None
        )

    # Tambah informasi preprocessing
    if col in preprocessing_metadata:

        column_info["preprocessing"] = preprocessing_metadata[col]

    data_dictionary["columns"].append(column_info)

# Simpan ke JSON

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(data_dictionary, f, indent=4, ensure_ascii=False)

# RIngkasan
print(f"CSV File      : {CSV_FILE}")
print(f"Rows          : {len(df)}")
print(f"Columns       : {len(df.columns)}")
print(f"Output File   : {OUTPUT_JSON}")

print("\Ringkasan Kolom:\n")

for col in data_dictionary["columns"]:

    print(f"- {col['column_name']} ({col['data_type']})")

    # Karena ada preprocessing
    if "preprocessing" in col:

        prep = col["preprocessing"]

        print(f"  preprocessing : {prep['method']}")
        print(f"  mapping       : {prep['mapping']}")

CSV File      : preprocessed_dataset.csv
Rows          : 11620
Columns       : 10
Output File   : data_dictionary.json
\Ringkasan Kolom:

- id (object)
- datetime (object)
- location (object)
- text (object)
- rating (int64)
- accessibility (int64)
  preprocessing : label_encoding
  mapping       : {'negative': 0, 'neutral': 1, 'positive': 2}
- facility (int64)
  preprocessing : label_encoding
  mapping       : {'negative': 0, 'neutral': 1, 'positive': 2}
- activity (int64)
  preprocessing : label_encoding
  mapping       : {'negative': 0, 'neutral': 1, 'positive': 2}
- word_count (int64)
- text_clean (object)


<>:105: SyntaxWarning: invalid escape sequence '\R'
<>:105: SyntaxWarning: invalid escape sequence '\R'
/tmp/ipykernel_14356/1420064010.py:105: SyntaxWarning: invalid escape sequence '\R'
  print("\Ringkasan Kolom:\n")
